# Database initialization

In [1]:
from db_management_tools import DataLoader
dataset_dir = r"D:\AIC26"
data_loader = DataLoader(dataset_dir)

Dataset record:
873 videos
59091 keyframes



# Upload data

In [2]:
data_loader.create_and_fill_database(
    "aic2026",
    upload_batch_size=1024,
    upload_num_workers=8
)

Deletion complete.


100%|██████████| 58/58 [31:33<00:00, 32.64s/it]


# Query initialization

In [3]:
from db_management_tools import QueryManager
from models import SigLIP2Model, BGEM3Model
from peripheral_tools import NotebookDisplay

image_text_feature_extractor = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
text_text_feature_extractor = BGEM3Model(name="BAAI/bge-m3", device="cuda")

query_manager = QueryManager(
    image_text_feature_extractor=image_text_feature_extractor,
    text_text_feature_extractor=text_text_feature_extractor
)
display = NotebookDisplay(315, 5)

[INFO] Loading quantized google/siglip2-so400m-patch14-384
[INFO] google/siglip2-so400m-patch14-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
from pprint import pprint
pprint(query_manager.describe_schema('aic26_mini'), indent=4)

In [ ]:
query = input("Enter your query: ")
limit = int(input("Enter your search limit: "))
points = query_manager.search(
    "aic26_mini",
    query=query,
    search_limit=limit,
    using_vectors=["siglip2_dense"]
)

display.display(points)

# Benchmark

In [ ]:
from peripheral_tools import Benchmarker, BenchmarkerV2

test_settings = [
    # {
    #     "using_vectors": ["siglip2_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_sparse"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense"]
    # },
    # {
    #     "using_vectors": ["siglip2_dense", "bgem3_dense", "bgem3_sparse"]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.7, 0.2, 0.1]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    # },
    {
        "use_aggregation": True,
        "aggregation_neighbor_limit": 10,
        "aggregation_weight": 0.5,
        "temporal_evaluation_weights": [0.6, 0.2, 0.2]
    },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.4,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
    # {
    #     "use_aggregation": True,
    #     "aggregation_neighbor_limit": 10,
    #     "aggregation_weight": 0.5,
    #     "temporal_evaluation_weights": [0.4, 0.3, 0.3]
    # },
]

benchmarker = BenchmarkerV2(query_manager)
res = benchmarker.do_benchmark(
    "aic26_mini",
    test_questions=r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\GITHUB_REPOS\Determinative-HCMCAIC26\question_bank\questions_with_answers.csv",
    test_settings=test_settings,
    save_record=False
)

Doing benchmark:   0%|          | 0/44 [00:00<?, ?it/s]











































Doing benchmark: 100%|██████████| 44/44 [01:06<00:00,  1.50s/it]

Benchmark results:
          Result@1  Result@5  Result@10  Result@50  mean_MRR  median_MRR  min_MRR  max_MRR   std_MRR  mean_Runtime  median_Runtime  min_Runtime  max_Runtime  std_Runtime
Method 1  0.545455  0.909091   0.931818   0.954545  0.687942         1.0      0.0      1.0  0.360739      1.498583        1.443522     1.279166     2.412586     0.190085
